# MILP'yi Değişken–Kısıt Bipartite Grafa Dönüştürmek

Bu notebook, küçük **0–1 MILP / binary packing** örneklerini değişken–kısıt (variable–constraint) bipartite grafa çevirip PyTorch Geometric ile bir GNN eğitir.

Amaç doğrudan "GNN solver yazmak" değildir. İki tip kullanım gösterilir:

1. **Warm-start / primal heuristic:** GNN her ikili değişken için `P(x_i=1)` üretir; tahmin feasibility repair'den geçirilip solver'a başlangıç çözümü olarak verilebilir.
2. **Branching için özellik/skor üretme:** LP relaxation'daki fractionality ile GNN belirsizliği birlikte kullanılarak aday değişkenler sıralanabilir. Bu, gerçek learned branching değildir; üretimde strong branching / pseudo-cost etiketleriyle eğitilmelidir.

Modellediğimiz problem:

\[
\max c^\top x
\]

\[
Ax \le b,\qquad x_i\in\{0,1\}
\]

Bipartite graph:

- `variable` düğümleri: karar değişkenleri,
- `constraint` düğümleri: doğrusal kısıtlar,
- `variable -> constraint` kenarı: \(A_{ji}\neq 0\),
- edge feature: \(A_{ji}\) katsayısı.

> Bu örnek eğitim amaçlı küçük instance'lar kullanır. Endüstriyel MILP öğrenmesinde SCIP/PySCIPOpt/Ecole üzerinden solver state feature'ları (reduced cost, dual, pseudo-cost, incumbent bilgisi vb.) alınmalıdır.


In [ ]:
import random
import itertools
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

from scipy.optimize import linprog
from torch_geometric.data import HeteroData
from torch_geometric.loader import DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


## 1. Küçük MILP instance'ları üretelim

Rastgele packing-type MILP'ler üretiyoruz. Boyut küçük tutulduğu için optimal ikili çözümü brute force ile hesaplayabiliyoruz; böylece GNN için öğretici etiket elde ediyoruz.

Ayrıca `scipy.optimize.linprog` ile LP relaxation çözüyoruz. LP çözümü gerçek MIP solver'lardaki önemli branching feature'larından biri olan **fractionality** bilgisini sağlar.


In [ ]:
def solve_binary_bruteforce(c, A, b):
    n = len(c)
    best_x = None
    best_obj = -np.inf

    for bits in itertools.product([0, 1], repeat=n):
        x = np.asarray(bits, dtype=float)
        if np.all(A @ x <= b + 1e-9):
            obj = float(c @ x)
            if obj > best_obj:
                best_obj = obj
                best_x = x.copy()

    return best_x, best_obj


def solve_lp_relaxation(c, A, b):
    res = linprog(
        -c,
        A_ub=A,
        b_ub=b,
        bounds=[(0.0, 1.0)] * len(c),
        method="highs",
    )
    if not res.success:
        raise RuntimeError(res.message)
    return res.x, float(c @ res.x)


def generate_instance(n_vars=8, n_cons=3, rng=None):
    rng = np.random.default_rng() if rng is None else rng

    c = rng.integers(1, 11, size=n_vars).astype(float)
    A = rng.integers(0, 6, size=(n_cons, n_vars)).astype(float)

    # Her değişken en az bir kısıta bağlı olsun.
    for j in range(n_vars):
        if np.all(A[:, j] == 0):
            A[rng.integers(0, n_cons), j] = rng.integers(1, 6)

    # Her kısıt anlamlı fakat uygulanabilir kalsın.
    row_sum = A.sum(axis=1)
    b = np.maximum(1.0, np.floor(row_sum * rng.uniform(0.35, 0.60, size=n_cons)))

    x_opt, obj_opt = solve_binary_bruteforce(c, A, b)
    x_lp, obj_lp = solve_lp_relaxation(c, A, b)

    return {
        "c": c,
        "A": A,
        "b": b,
        "x_opt": x_opt,
        "obj_opt": obj_opt,
        "x_lp": x_lp,
        "obj_lp": obj_lp,
    }


example = generate_instance(rng=np.random.default_rng(SEED))
example


## 2. MILP → `HeteroData`

Variable node feature'ları:

- normalize edilmiş amaç katsayısı \(c_i\),
- LP relaxation değeri \(x_i^{LP}\),
- LP fractionality,
- variable degree.

Constraint node feature'ları:

- normalize RHS,
- LP slack,
- constraint degree.

Edge feature:

- normalize \(A_{ji}\) katsayısı.

Gerçek bir SCIP entegrasyonunda bunlara reduced cost, dual değer, basis status, pseudo-cost, incumbent ilişkisi ve bound bilgileri eklenebilir.


In [ ]:
def safe_scale(x):
    x = np.asarray(x, dtype=float)
    m = np.max(np.abs(x))
    return x / m if m > 0 else x


def milp_to_heterodata(inst):
    c, A, b = inst["c"], inst["A"], inst["b"]
    x_lp, x_opt = inst["x_lp"], inst["x_opt"]

    n_cons, n_vars = A.shape

    var_degree = (A != 0).sum(axis=0).astype(float) / max(1, n_cons)
    fractionality = 1.0 - 2.0 * np.abs(x_lp - 0.5)
    fractionality = np.clip(fractionality, 0.0, 1.0)

    var_x = np.column_stack([
        safe_scale(c),
        x_lp,
        fractionality,
        var_degree,
    ])

    lp_slack = b - A @ x_lp
    con_degree = (A != 0).sum(axis=1).astype(float) / max(1, n_vars)
    con_x = np.column_stack([
        b / np.maximum(A.sum(axis=1), 1.0),
        lp_slack / np.maximum(b, 1.0),
        con_degree,
    ])

    rows, cols = np.nonzero(A)
    edge_index = np.vstack([cols, rows])  # variable -> constraint
    coeff = A[rows, cols]
    coeff = coeff / max(np.max(np.abs(A)), 1.0)

    data = HeteroData()
    data["variable"].x = torch.tensor(var_x, dtype=torch.float32)
    data["variable"].y = torch.tensor(x_opt, dtype=torch.float32)
    data["constraint"].x = torch.tensor(con_x, dtype=torch.float32)

    data["variable", "participates", "constraint"].edge_index = torch.tensor(
        edge_index, dtype=torch.long
    )
    data["variable", "participates", "constraint"].edge_attr = torch.tensor(
        coeff[:, None], dtype=torch.float32
    )

    return data


graph = milp_to_heterodata(example)
graph


## 3. Katsayı-duyarlı bipartite message passing

Hazır bir GCN katmanını doğrudan kullanmak yerine edge coefficient \(A_{ji}\)'yi mesajın içine katıyoruz.

İki yönlü akış:

```text
variable --(A_ji)--> constraint
constraint --(A_ji)--> variable
```

Bu tasarım, MILP graph'larında neden "heterogeneous/bipartite GNN" dediğimizi somutlaştırır.


In [ ]:
def mean_aggregate(messages, index, dim_size):
    out = messages.new_zeros((dim_size, messages.size(-1)))
    out.index_add_(0, index, messages)

    count = messages.new_zeros((dim_size, 1))
    ones = messages.new_ones((messages.size(0), 1))
    count.index_add_(0, index, ones)

    return out / count.clamp_min(1.0)


class BipartiteBlock(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.var_to_con = nn.Sequential(
            nn.Linear(hidden_dim + 1, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
        self.update_con = nn.Sequential(
            nn.Linear(2 * hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

        self.con_to_var = nn.Sequential(
            nn.Linear(hidden_dim + 1, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
        self.update_var = nn.Sequential(
            nn.Linear(2 * hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

        self.var_norm = nn.LayerNorm(hidden_dim)
        self.con_norm = nn.LayerNorm(hidden_dim)

    def forward(self, h_var, h_con, edge_index, edge_attr):
        var_idx, con_idx = edge_index

        msg_vc = self.var_to_con(torch.cat([h_var[var_idx], edge_attr], dim=-1))
        agg_con = mean_aggregate(msg_vc, con_idx, h_con.size(0))
        h_con = self.con_norm(
            h_con + self.update_con(torch.cat([h_con, agg_con], dim=-1))
        )

        msg_cv = self.con_to_var(torch.cat([h_con[con_idx], edge_attr], dim=-1))
        agg_var = mean_aggregate(msg_cv, var_idx, h_var.size(0))
        h_var = self.var_norm(
            h_var + self.update_var(torch.cat([h_var, agg_var], dim=-1))
        )

        return h_var, h_con


class MILPBipartiteGNN(nn.Module):
    def __init__(self, hidden_dim=64, n_layers=3):
        super().__init__()
        self.var_encoder = nn.Linear(4, hidden_dim)
        self.con_encoder = nn.Linear(3, hidden_dim)
        self.blocks = nn.ModuleList(
            [BipartiteBlock(hidden_dim) for _ in range(n_layers)]
        )
        self.var_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, data):
        h_var = F.relu(self.var_encoder(data["variable"].x))
        h_con = F.relu(self.con_encoder(data["constraint"].x))

        relation = data["variable", "participates", "constraint"]
        for block in self.blocks:
            h_var, h_con = block(
                h_var,
                h_con,
                relation.edge_index,
                relation.edge_attr,
            )

        return self.var_head(h_var).squeeze(-1)


## 4. Eğitim veri kümesi

Her graph başka bir MILP instance'ını temsil eder. Böylece model tek bir problemi ezberlemek yerine instance'lar arası yapısal örüntü öğrenmeye çalışır.

Bu yine de küçük bir demo'dur. Ciddi deneyde:

- train/test problem boyutları ayrılmalı,
- farklı instance dağılımlarında OOD test yapılmalı,
- solver runtime ve node count ölçülmeli,
- yalnız classification accuracy raporlanmamalıdır.


In [ ]:
rng = np.random.default_rng(SEED)

train_raw = [generate_instance(rng=rng) for _ in range(300)]
test_raw = [generate_instance(rng=rng) for _ in range(60)]

train_graphs = [milp_to_heterodata(x) for x in train_raw]
test_graphs = [milp_to_heterodata(x) for x in test_raw]

train_loader = DataLoader(train_graphs, batch_size=32, shuffle=True)
test_loader = DataLoader(test_graphs, batch_size=32, shuffle=False)

model = MILPBipartiteGNN(hidden_dim=64, n_layers=3).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)

for epoch in range(1, 61):
    model.train()
    total_loss = 0.0
    n_items = 0

    for batch in train_loader:
        batch = batch.to(device)
        logits = model(batch)
        target = batch["variable"].y

        loss = F.binary_cross_entropy_with_logits(logits, target)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += float(loss) * target.numel()
        n_items += target.numel()

    if epoch == 1 or epoch % 10 == 0:
        print(f"epoch={epoch:02d}  loss={total_loss / n_items:.4f}")


## 5. Warm-start üretme ve feasibility repair

Threshold ile elde edilen ikili tahmin her zaman feasible olmayabilir. Bu yüzden küçük bir repair heuristic kullanıyoruz.

Gerçek solver akışında daha güçlü seçenekler:

- MIP start,
- feasibility pump,
- local branching,
- RINS,
- large neighborhood search,
- solver'ın kendi primal repair mekanizmaları.


In [ ]:
def repair_packing(x, c, A, b):
    x = np.asarray(x, dtype=int).copy()

    while np.any(A @ x > b + 1e-9):
        selected = np.flatnonzero(x == 1)
        if len(selected) == 0:
            break

        # Kaynak kullanımına göre en az verimli seçili değişkeni çıkar.
        resource = A[:, selected].sum(axis=0)
        efficiency = c[selected] / np.maximum(resource, 1e-9)
        remove = selected[np.argmin(efficiency)]
        x[remove] = 0

    return x


@torch.no_grad()
def predict_instance(inst):
    data = milp_to_heterodata(inst).to(device)
    logits = model(data)
    prob = torch.sigmoid(logits).cpu().numpy()

    raw = (prob >= 0.5).astype(int)
    repaired = repair_packing(raw, inst["c"], inst["A"], inst["b"])

    return prob, raw, repaired


inst = test_raw[0]
prob, raw, repaired = predict_instance(inst)

print("c =", inst["c"].astype(int))
print("LP relaxation =", np.round(inst["x_lp"], 3))
print("GNN P(x_i=1) =", np.round(prob, 3))
print("Ham threshold =", raw)
print("Repair sonrası =", repaired)
print("Optimal çözüm =", inst["x_opt"].astype(int))
print()
print("Warm-start objective =", float(inst["c"] @ repaired))
print("Optimal objective    =", inst["obj_opt"])


## 6. Test kümesinde objective gap

Bir learned heuristic'i **accuracy** ile değil, optimizasyon metriğiyle değerlendirmek gerekir.

Burada:

\[
gap = \frac{z^\* - z_{GNN}}{\max(|z^\*|, \epsilon)}
\]

hesaplıyoruz.

Production değerlendirmesinde ayrıca:

- solver wall-clock time,
- branch-and-bound node sayısı,
- time-to-first-feasible,
- primal/dual integral,
- optimality gap'in zamana göre seyri

ölçülmelidir.


In [ ]:
gaps = []
feasible_count = 0

for inst in test_raw:
    prob, raw, repaired = predict_instance(inst)
    feasible = np.all(inst["A"] @ repaired <= inst["b"] + 1e-9)
    feasible_count += int(feasible)

    obj = float(inst["c"] @ repaired)
    gap = (inst["obj_opt"] - obj) / max(abs(inst["obj_opt"]), 1e-9)
    gaps.append(gap)

print(f"Feasible warm-start oranı: {feasible_count / len(test_raw):.1%}")
print(f"Ortalama objective gap   : {np.mean(gaps):.2%}")
print(f"Medyan objective gap     : {np.median(gaps):.2%}")


## 7. Branching skoru fikri

Aşağıdaki skor **gerçek learned branching değildir**. Yalnızca iki sinyali birleştiren öğretici bir örnektir:

- LP fractionality: \(x_i^{LP}\) değeri 0.5'e ne kadar yakın?
- GNN uncertainty: \(P(x_i=1)\) değeri 0.5'e ne kadar yakın?

Branch-and-bound'da aday değişkenler arasında yüksek skor alanlar önce incelenebilir.

Gerçek bir çalışma için hedef değişken `strong branching score`, `pseudo-cost` veya solver'ın seçtiği branch değişkeni olabilir; GNN bu etiketi imitation learning ile öğrenir.


In [ ]:
def branching_priority(lp_x, prob):
    lp_fractionality = 1.0 - 2.0 * np.abs(lp_x - 0.5)
    gnn_uncertainty = 1.0 - 2.0 * np.abs(prob - 0.5)

    lp_fractionality = np.clip(lp_fractionality, 0.0, 1.0)
    gnn_uncertainty = np.clip(gnn_uncertainty, 0.0, 1.0)

    return 0.6 * lp_fractionality + 0.4 * gnn_uncertainty


scores = branching_priority(inst["x_lp"], prob)
ranking = np.argsort(-scores)

for rank, j in enumerate(ranking, start=1):
    print(
        f"{rank:2d}. x_{j}: score={scores[j]:.3f}, "
        f"LP={inst['x_lp'][j]:.3f}, P(1)={prob[j]:.3f}"
    )


## 8. Bunu gerçek SCIP/PySCIPOpt çalışmasına nasıl taşırız?

Bu notebook'taki akış:

```text
MILP instance
    ↓
variable-constraint bipartite graph
    ↓
GNN
    ↓
P(x_i = 1)
    ↓
repair
    ↓
warm start
```

Gerçek **learned branching** akışı ise:

```text
SCIP branch-and-bound node'u
    ↓
LP relaxation + solver feature'ları
    ↓
variable-constraint bipartite graph
    ↓
GNN
    ↓
candidate variable scores
    ↓
SCIP branching rule
```

Önerilen stack:

```text
PyTorch
PyTorch Geometric
PySCIPOpt
SCIP
Ecole (uygun deneylerde)
```

Önemli ayrım:

- GNN'nin tahmini **doğruluk garantisi** değildir.
- Solver feasibility ve optimality mekanizmasını korur.
- GNN'nin görevi arama maliyetini azaltmak veya iyi primal çözüme daha erken ulaşmaktır.

Bir sonraki doğal notebook, `PySCIPOpt` kullanarak gerçek bir branching callback/rule içinde GNN skorlarının nasıl çağrılacağını göstermektir.
